# Практика · BERT у роботі

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

Цей зошит бере передтреновану мовну модель і **застосовує** її до справжньої задачі
трьома різними способами. Він самодостатній: усе, що потрібно, тут пояснено.

Що зробимо:

1. Зберемо корпус українських перекладів і поставимо мітку «це повідомлення про помилку».
2. Навчимо власну **масковану мовну модель** — маленький двошаровий енкодер.
3. Побудуємо на ній три способи переносу: **лінійний зонд**, **часткове** й **повне донавчання**.
4. Доберемо швидкість навчання кожному способу окремо, на **відкладеній** вибірці.
5. Порівняємо всі три на пʼятьох зернах і подивимось на **купи**, а не на середні.
6. Заміряємо, як донавчання **забуває** мову, і як розкид залежить від кількості міток.
7. Зʼясуємо, який шар вигідно розморожувати, і що знає кожен шар.
8. Поставимо поруч рубіж без нейромереж: TF-IDF із логістичною регресією.

> ⏱ Зошит навчає одну модель із нуля й проганяє понад пʼятдесят донавчань.
> Заміряно: близько **чотирьох з половиною хвилин** процесорного часу
> на чотирьох ядрах без відеокарти. Точне число надрукує остання клітинка.

## 1 · Середовище

Перша клітинка просить числові бібліотеки рахувати в **один потік**. Це не оптимізація,
а умова того, щоб замір часу взагалі щось означав: на завантаженій машині потоки
більшу частину часу чекають одне на одного, і це очікування записується в процесорний
час. Ставити змінні треба **до** імпорту numpy — пізніше вже не подіє.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys, re, glob, gettext, time, random, collections, gc
import numpy as np
import torch
import torch.nn as nn
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

torch.set_num_threads(1)          # те саме прохання, але вже до torch

START = time.process_time()       # процесорний час усього зошита

print("Python ", sys.version.split()[0])
print("torch  ", torch.__version__)
print("numpy  ", np.__version__)
print("sklearn", sklearn.__version__)
print("потоків:", torch.get_num_threads())

## 2 · Корпус: українські переклади інтерфейсів

Дані справжні й лежать просто в системі: файли `.mo` української локалі містять пари
«англійський оригінал → український переклад». Ознаки братимемо з **українського**
боку, а мітку — з **англійського**. Так модель не бачить того тексту, з якого зроблено
мітку.

⚠️ **Без української локалі зошит не виконається.** Йому потрібні сотні тисяч речень,
і жоден вбудований мінікорпус їх не замінить. Перевірити, чи є корпус:
`ls /usr/share/locale/uk/LC_MESSAGES/*.mo | wc -l` — потрібно приблизно від сотні файлів.

In [ ]:
TOKEN_PATTERN = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")   # канон блоку
ERROR_WORDS = re.compile(r"\b(error|fail|failed|cannot|unable|invalid|denied|corrupt)\b",
                         re.IGNORECASE)

def load_pairs():
    """Читаємо .mo-файли й повертаємо пари (англійський оригінал, український переклад)."""
    pairs = []
    for path in sorted(glob.glob("/usr/share/locale/uk/LC_MESSAGES/*.mo")):
        try:
            with open(path, "rb") as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # зламаний або чужий формат — пропускаємо
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і текстом не є
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and "Project-Id" not in target:
                pairs.append((source, target))
    return pairs

pairs = load_pairs()
if len(pairs) < 20000:
    raise RuntimeError(
        "Українська локаль не знайдена або надто мала: %d пар. "
        "Зошиту потрібні сотні тисяч речень." % len(pairs))

print("пар «оригінал → переклад»:", len(pairs))
print()
print("приклад:")
print("  англ:", pairs[0][0][:70])
print("  укр :", pairs[0][1][:70])

## 3 · Мітка, поділ і словник

Мітка — «чи повідомляє цей рядок про помилку». Її ставить регулярка по англійському
оригіналу: вісім слів, якими англомовні програми повідомляють про збій.

Три речі, які тут важливі:

- **Речення перемішуємо перед поділом.** Корпус лежить у порядку програм: спершу всі
  рядки однієї, потім іншої. Узяти «перші 10 %» означало б узяти не менше даних, а
  вужчий домен.
- **Ділимо на три частини, а не на дві.** Навчальна — вчити, **відкладена** — обирати
  налаштування, перевірна — назвати число один раз наприкінці. Якби ми добирали крок
  на перевірній, то підглядали б у відповідь.
- **Словник рахуємо лише по навчальній частині.** Слова, які трапились менш ніж пʼять
  разів, замінюємо на `<unk>`: побачене раз слово не дає моделі нічого, крім змоги
  запамʼятати один документ.

In [ ]:
MAXLEN = 24            # довші речення обрізаємо
rows = []
for source, target in pairs:
    words = TOKEN_PATTERN.findall(target.lower())
    if 3 <= len(words) <= 30:                 # зовсім куці й зовсім довгі відкидаємо
        rows.append((words, 1 if ERROR_WORDS.search(source) else 0))

random.Random(0).shuffle(rows)                # грабля: корпус лежить у порядку програм
n_rows = len(rows)
cut_train, cut_hold = int(0.8 * n_rows), int(0.9 * n_rows)

counts = collections.Counter(w for words, _ in rows[:cut_train] for w in words)
itos = ["<pad>", "<mask>", "<unk>", "<cls>"] + [w for w, c in counts.most_common() if c >= 5]
stoi = {w: i for i, w in enumerate(itos)}
VOCAB = len(itos)
PAD, MASK, UNK, CLS = 0, 1, 2, 3

def encode(words):
    """Речення -> список номерів токенів, із [CLS] на початку."""
    return [CLS] + [stoi.get(w, UNK) for w in words][:MAXLEN - 1]

ids = [encode(words) for words, _ in rows]
labels = [y for _, y in rows]

train_x, train_y = ids[:cut_train], labels[:cut_train]
hold_x,  hold_y  = ids[cut_train:cut_hold], labels[cut_train:cut_hold]
test_x,  test_y  = ids[cut_hold:], labels[cut_hold:]

SHARE = sum(labels) / len(labels)
print("прикладів усього :", n_rows)
print("навчальних       :", len(train_x))
print("відкладених      :", len(hold_x))
print("перевірних       :", len(test_x))
print("словник          :", VOCAB)
print("частка «помилка» : %.4f" % SHARE)

Клас «помилка» рідкісний — приблизно кожен пʼятий приклад. Якщо цього не врахувати,
модель швидко навчиться відповідати «не помилка» завжди: так вона матиме високу
точність і нульовий F1. Тому в усіх навчаннях зошита ми **зважуємо втрату**: помилка
на рідкісному класі коштує стільки, у скільки разів він рідкісніший.

Ще нам знадобиться дві дрібні службові функції: доповнення батча нулями до
однакової довжини й підрахунок F1 для класу «помилка».

In [ ]:
CLASS_WEIGHT = torch.tensor([1.0, (1 - SHARE) / SHARE])
print("вага класу «помилка»: %.4f" % float(CLASS_WEIGHT[1]))

def pad_batch(batch):
    """Список списків різної довжини -> прямокутний тензор, порожнє місце = <pad>."""
    width = max(len(x) for x in batch)
    return torch.tensor([x + [PAD] * (width - len(x)) for x in batch])

def f1_error(predicted, gold):
    """F1 для класу «помилка» (мітка 1)."""
    tp = sum(1 for p, g in zip(predicted, gold) if p == 1 and g == 1)
    fp = sum(1 for p, g in zip(predicted, gold) if p == 1 and g == 0)
    fn = sum(1 for p, g in zip(predicted, gold) if p == 0 and g == 1)
    return 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)

# рівно ті шматки, на яких звітуємо: відкладена — для добору, перевірна — для підсумку
HOLD_EVAL = (hold_x[:1000], hold_y[:1000])
TEST_EVAL = (test_x[:3000], test_y[:3000])
print("для добору беремо %d відкладених, для звіту %d перевірних"
      % (len(HOLD_EVAL[0]), len(TEST_EVAL[0])))

## 4 · Тіло моделі

Тіло — це те, що передтреновують і потім переносять. Воно складається з трьох частин:

- **таблиця ембедингів слів** — по вектору на кожне слово словника;
- **таблиця ембедингів позицій** — по вектору на кожне місце в реченні, щоб модель
  розрізняла порядок слів;
- **стос енкодерів** — два шари, кожен зі своєю увагою й подавальною мережею.

На вхід іде список номерів токенів, на виході — **по вектору на кожну позицію**.
Метод `forward` уміє повернути не лише останній шар, а й усі проміжні: далі ми
дивитимемось, що знає кожен із них.

In [ ]:
DIM = 128           # розмір вектора
HEADS = 4           # голів уваги
LAYERS = 2          # шарів енкодера

class Encoder(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.emb = nn.Embedding(vocab, DIM, padding_idx=PAD)
        self.pos = nn.Embedding(MAXLEN, DIM)
        layer = nn.TransformerEncoderLayer(DIM, HEADS, dim_feedforward=4 * DIM,
                                           batch_first=True, dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, LAYERS, enable_nested_tensor=False)

    def forward(self, x, collect=False):
        h = self.emb(x) + self.pos(torch.arange(x.size(1)))
        keypad = (x == PAD)                 # порожні місця увага має ігнорувати
        stages = [h]                        # stages[0] — до першого шару енкодера
        for layer in self.body.layers:
            h = layer(h, src_key_padding_mask=keypad)
            stages.append(h)
        return stages if collect else h

torch.manual_seed(0)
probe_model = Encoder(VOCAB)
n_emb = sum(p.numel() for p in probe_model.emb.parameters()) + \
        sum(p.numel() for p in probe_model.pos.parameters())
n_layer = sum(p.numel() for p in probe_model.body.layers[0].parameters())
n_total = sum(p.numel() for p in probe_model.parameters())
print("ваг у таблицях ембедингів :", n_emb)
print("ваг в одному шарі енкодера:", n_layer)
print("ваг у тілі разом          :", n_total)
# перевіряємо власну арифметику: ембединги + два однакові шари = усе тіло
assert n_emb + 2 * n_layer == n_total, "порахували не всі ваги!"
del probe_model
_ = gc.collect()
print("✅ ембединги + два шари справді дають усе тіло")

## 5 · Передтренування: маскована мовна модель

Тут беруться мітки задарма. Ми ховаємо приблизно кожне сьоме слово речення під
токеном `<mask>` і просимо модель відгадати, що там стояло. Правильна відповідь відома
з самого тексту — розмічати нічого не треба.

Зверху на тіло ставимо **голову MLM**: лінійний шар зі 128 чисел у розмір словника.
Рахуємо її **лише на схованих позиціях**: тільки там є що передбачати, і це втричі
дешевше, ніж рахувати на всіх.

Втрата — перехресна ентропія. Орієнтир: модель, яка нічого не знає й вгадує рівномірно
зі словника, дала б `ln(VOCAB)`.

In [ ]:
PRETRAIN_STEPS = 500
BATCH = 64

torch.manual_seed(0)
body = Encoder(VOCAB)
mlm_head = nn.Linear(DIM, VOCAB)
optimizer = torch.optim.Adam(list(body.parameters()) + list(mlm_head.parameters()), lr=2e-3)
picker = random.Random(0)
all_train = list(range(len(train_x)))
history = []

t0 = time.process_time()
for step in range(PRETRAIN_STEPS):
    x = pad_batch([train_x[i] for i in picker.sample(all_train, BATCH)])
    target = x.clone()
    # ховаємо 15 % позицій, але не заповнювач і не [CLS]
    hidden = (torch.rand(x.shape) < 0.15) & (x != PAD) & (x != CLS)
    if hidden.sum() == 0:
        continue
    masked_input = x.clone()
    masked_input[hidden] = MASK
    loss = nn.functional.cross_entropy(mlm_head(body(masked_input)[hidden]), target[hidden])
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append(loss.item())
pretrain_sec = time.process_time() - t0
del optimizer; gc.collect()

print("кроків              :", PRETRAIN_STEPS)
print("процесорного часу   : %.1f с  (%.4f с на крок)" % (pretrain_sec, pretrain_sec / PRETRAIN_STEPS))
print("втрата, перші 50    : %.4f" % np.mean(history[:50]))
print("втрата, останні 50  : %.4f" % np.mean(history[-50:]))
print("вгадування навмання : %.4f" % np.log(VOCAB))

Зберігаємо ваги передтренованого тіла й голови MLM. Ці ваги — і є те, що ми далі
переноситимемо: кожен спосіб починатиме з них, а не з випадкового шуму.

In [ ]:
PRETRAINED = {k: v.detach().clone() for k, v in body.state_dict().items()}
PRETRAINED_MLM = {k: v.detach().clone() for k, v in mlm_head.state_dict().items()}

def mlm_loss(encoder, head, sentences, seed=7):
    """Наскільки добре модель відновлює сховані слова. Маска фіксована зерном,
    щоб два заміри були зіставні між собою."""
    gen = torch.Generator().manual_seed(seed)
    total, batches = 0.0, 0
    encoder.eval()
    with torch.no_grad():
        for i in range(0, len(sentences), 256):
            x = pad_batch(sentences[i:i + 256])
            target = x.clone()
            hidden = (torch.rand(x.shape, generator=gen) < 0.15) & (x != PAD) & (x != CLS)
            if hidden.sum() == 0:
                continue
            masked_input = x.clone(); masked_input[hidden] = MASK
            total += float(nn.functional.cross_entropy(head(encoder(masked_input)[hidden]),
                                                       target[hidden]))
            batches += 1
    encoder.train()
    return total / batches

MLM_BEFORE = mlm_loss(body, mlm_head, hold_x[:1024])
print("втрата MLM передтренованої моделі на відкладеній: %.4f" % MLM_BEFORE)
print("те саме для рівномірного вгадування             : %.4f" % np.log(VOCAB))
print("тобто передтренування зняло %.4f пункту" % (np.log(VOCAB) - MLM_BEFORE))

## 6 · Три способи взяти цю модель

Тепер головна конструкція зошита. Функція нижче збирає модель під задачу класифікації
й розставляє **замки**: `requires_grad = False` означає «цю вагу не змінюємо».

Способи:

- `probe` — заморожене все тіло, вчиться лише лінійна голова;
- `partial` — заморожені ембединги й нижній шар, вчиться верхній шар і голова;
- `full` — вчиться все;
- `low` і `emb` — два додаткові варіанти для розділу «який шар розморожувати».

Зверни увагу на єдину відмінність у циклі навчання: для `probe` прямий прохід тіла
загорнуто в `torch.no_grad()`. Це не оптимізація заради краси — це те, що робить зонд
дешевим: граф для зворотного проходу через тіло навіть не будується.

In [ ]:
def build(mode, seed, pretrained=True):
    """Повертає (тіло, голова, список ваг, які вчаться)."""
    torch.manual_seed(seed)
    encoder = Encoder(VOCAB)
    if pretrained:
        encoder.load_state_dict(PRETRAINED)
    classifier = nn.Linear(DIM, 2)

    if mode == "full":
        trainable = list(encoder.parameters()) + list(classifier.parameters())
    else:
        for p in encoder.parameters():
            p.requires_grad = False          # спершу замикаємо все
        if mode == "partial":
            unlocked = encoder.body.layers[-1].parameters()
        elif mode == "low":
            unlocked = encoder.body.layers[0].parameters()
        elif mode == "emb":
            unlocked = encoder.emb.parameters()
        else:                                 # probe — не відмикаємо нічого
            unlocked = []
        for p in unlocked:
            p.requires_grad = True
        trainable = [p for p in encoder.parameters() if p.requires_grad] \
                    + list(classifier.parameters())
    return encoder, classifier, trainable

for mode in ("probe", "partial", "full"):
    encoder, classifier, trainable = build(mode, 0)
    n = sum(p.numel() for p in trainable)
    print("%-8s вчиться %9d ваг  (%6.2f %% моделі)" % (mode, n, 100 * n / (n_total + 258)))
    del encoder, classifier, trainable
    _ = gc.collect()

Цикл донавчання — один на всі способи. Однаковий бюджет (сто кроків батчем 32),
однакова зважена втрата, однакове зерно для вибору розмічених прикладів. Різниця
тільки в замках.

In [ ]:
STEPS = 100
FT_BATCH = 32

def finetune(mode, n_labels, lr, seed, evalset, measure_mlm=False):
    """Донавчаємо й повертаємо (F1, процесорний час навчання, втрата MLM після, ваг)."""
    encoder, classifier, trainable = build(mode, seed)
    n_weights = sum(p.numel() for p in trainable)
    optimizer = torch.optim.Adam(trainable, lr=lr)
    picker = random.Random(1000 + seed)
    chosen = picker.sample(range(len(train_x)), n_labels)   # випадкова частка, не префікс
    xs = [train_x[i] for i in chosen]
    ys = [train_y[i] for i in chosen]

    t0 = time.process_time()
    for step in range(STEPS):
        batch = [picker.randrange(len(xs)) for _ in range(min(FT_BATCH, len(xs)))]
        x = pad_batch([xs[i] for i in batch])
        y = torch.tensor([ys[i] for i in batch])
        if mode == "probe":
            with torch.no_grad():            # тіло заморожене — граф не потрібен
                vector = encoder(x)[:, 0]
            logits = classifier(vector)
        else:
            logits = classifier(encoder(x)[:, 0])
        loss = nn.functional.cross_entropy(logits, y, weight=CLASS_WEIGHT)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    train_sec = time.process_time() - t0

    encoder.eval()
    predicted = []
    with torch.no_grad():
        for i in range(0, len(evalset[0]), 256):
            batch_x = pad_batch(evalset[0][i:i + 256])
            predicted += classifier(encoder(batch_x)[:, 0]).argmax(-1).tolist()
    score = f1_error(predicted, evalset[1])

    after = None
    if measure_mlm:
        head_copy = nn.Linear(DIM, VOCAB)
        head_copy.load_state_dict(PRETRAINED_MLM)   # голова MLM та сама, змінилось тіло
        after = mlm_loss(encoder, head_copy, hold_x[:1024])
        del head_copy
    del encoder, classifier, trainable, optimizer
    gc.collect()                                    # один важкий прогін за раз
    return score, train_sec, after, n_weights

print("бюджет донавчання: %d кроків батчем %d" % (STEPS, FT_BATCH))
print("тобто модель побачить %d прикладів за прогін" % (STEPS * FT_BATCH))

Перш ніж міряти якість, переконаймось, що замки справді тримають. Найпряміший спосіб:
донавчити зонд і побітово звірити ваги тіла з тими, що були до навчання.

In [ ]:
encoder, classifier, trainable = build("probe", 0)
head_before = classifier.weight.detach().clone()
optimizer = torch.optim.Adam(trainable, lr=1e-2)
for step in range(20):
    x = pad_batch(train_x[:32]); y = torch.tensor(train_y[:32])
    with torch.no_grad():
        vector = encoder(x)[:, 0]
    loss = nn.functional.cross_entropy(classifier(vector), y, weight=CLASS_WEIGHT)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

body_same = all(torch.equal(v, PRETRAINED[k]) for k, v in encoder.state_dict().items())
head_moved = float((classifier.weight - head_before).abs().max())
assert body_same, "заморожене тіло змінилось — замки не тримають!"
assert head_moved > 0, "голова не зрушила — навчання не відбулось!"
print("✅ після 20 кроків зонда всі ваги тіла побітово ті самі")
print("   а ваги голови зрушили щонайбільше на %.4f — вчилась саме вона" % head_moved)
del encoder, classifier, trainable, optimizer, head_before
_ = gc.collect()

## 7 · Добір швидкості навчання

Порівнювати способи однією швидкістю навчання було б нечесно: зонд учить 258 ваг і
любить великий крок, повне донавчання вчить мільйон і від такого кроку розсипається.
Тому крок добираємо **кожному способу окремо**, на однаковій сітці з пʼяти значень,
на **відкладеній** вибірці й **тим самим бюджетом**, з яким потім працюватимемо.

Для повного донавчання додатково міряємо втрату MLM після кожного прогону — це й буде
замір **катастрофічного забування**.

In [ ]:
GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
best_lr, grid_rows = {}, {}

for mode in ("probe", "partial", "full"):
    scores, row = [], []
    for lr in GRID:
        runs, forgets = [], []
        for seed in (0, 1):                 # два зерна, щоб вибір не був випадковим
            score, sec, after, _ = finetune(mode, 2000, lr, seed, HOLD_EVAL,
                                            measure_mlm=(mode == "full"))
            runs.append(score)
            if after is not None:
                forgets.append(after)
        mean_score = float(np.mean(runs))
        mean_mlm = float(np.mean(forgets)) if forgets else None
        scores.append(mean_score)
        row.append((lr, mean_score, mean_mlm))
        print("  %-8s lr=%-7g відкладена F1 %.4f%s"
              % (mode, lr, mean_score,
                 "   втрата MLM після %.4f" % mean_mlm if mean_mlm else ""))
    k = int(np.argmax(scores))
    best_lr[mode] = GRID[k]
    grid_rows[mode] = row
    print("  -> %s: беремо lr=%g%s\n"
          % (mode, GRID[k], "  ⚠️ КРАЙ СІТКИ" if k in (0, len(GRID) - 1) else ""))

Ось таблиця забування окремо — вона того варта. Ліворуч якість на нашій задачі,
праворуч — чи памʼятає модель мову після донавчання.

In [ ]:
print("до донавчання втрата MLM була %.4f (навмання дало б %.4f)\n"
      % (MLM_BEFORE, np.log(VOCAB)))
print("%-10s %-12s %-14s %s" % ("крок", "F1 задачі", "втрата MLM", "мова"))
for lr, score, after in grid_rows["full"]:
    if after < MLM_BEFORE * 1.02:
        verdict = "ціла"
    elif after < np.log(VOCAB):
        verdict = "потерпіла"
    else:
        verdict = "знищена"
    print("%-10g %-12.4f %-14.4f %s" % (lr, score, after, verdict))

## 8 · Головне порівняння: три способи, пʼять зерен

Тепер порівняння на перевірній вибірці. Пʼять зерен — і дивимось на **купу** (від
найгіршого запуску до найкращого), а не на середнє: різниця, менша за розкид, не є
різницею.

In [ ]:
LABELS = 2000
main = {}
for mode in ("probe", "partial", "full"):
    scores, times, weights = [], [], 0
    for seed in range(5):
        score, sec, _, weights = finetune(mode, LABELS, best_lr[mode], seed, TEST_EVAL)
        scores.append(score); times.append(sec)
    main[mode] = dict(f1=scores, median=float(np.median(scores)),
                      lo=min(scores), hi=max(scores),
                      sec=float(np.mean(times)), weights=weights)
    print("%-8s lr=%-7g медіана %.4f  купа %.4f…%.4f  %.2f с/прогін  %d ваг"
          % (mode, best_lr[mode], main[mode]["median"], main[mode]["lo"],
             main[mode]["hi"], main[mode]["sec"], weights))
    print("         зерна:", " ".join("%.4f" % s for s in scores))
print()
for a, b in (("probe", "partial"), ("partial", "full")):
    overlap = not (main[a]["lo"] > main[b]["hi"] or main[b]["lo"] > main[a]["hi"])
    print("купи «%s» і «%s» %s" % (a, b, "перетинаються" if overlap else "НЕ перетинаються"))

## 9 · Нестабільність на малому наборі

Та сама конфігурація, той самий крок, той самий бюджет — міняється лише кількість
розмічених прикладів. Пʼять зерен.

In [ ]:
small = []
for seed in range(5):
    score, _, _, _ = finetune("full", 300, best_lr["full"], seed, TEST_EVAL)
    small.append(score)
small_width = max(small) - min(small)
big_width = main["full"]["hi"] - main["full"]["lo"]

print("300 міток :", " ".join("%.4f" % s for s in small))
print("   медіана %.4f  купа %.4f…%.4f  ширина %.4f"
      % (np.median(small), min(small), max(small), small_width))
print("2000 міток:", " ".join("%.4f" % s for s in main["full"]["f1"]))
print("   медіана %.4f  купа %.4f…%.4f  ширина %.4f"
      % (main["full"]["median"], main["full"]["lo"], main["full"]["hi"], big_width))
print()
print("купа на 300 мітках ширша в %.2f раза" % (small_width / big_width))

## 10 · Який шар розморожувати

Часткове донавчання ставить конкретне питання: якщо можна відімкнути рівно одну
частину моделі, яку брати? Три конфігурації, однаковий бюджет, однаковий крок,
три зерна. Варіант «верхній шар» — це та сама гілка `partial`, тож беремо її перші
три зерна, щоб не рахувати те саме двічі.

In [ ]:
unfreeze = {"верхній шар": (main["partial"]["f1"][:3], main["partial"]["weights"])}
for mode, title in (("low", "нижній шар"), ("emb", "ембединги")):
    scores, weights = [], 0
    for seed in range(3):
        score, _, _, weights = finetune(mode, LABELS, best_lr["partial"], seed, TEST_EVAL)
        scores.append(score)
    unfreeze[title] = (scores, weights)

for title in ("нижній шар", "верхній шар", "ембединги"):
    scores, weights = unfreeze[title]
    print("%-12s %8d ваг   медіана %.4f  купа %.4f…%.4f   зерна %s"
          % (title, weights, np.median(scores), min(scores), max(scores),
             " ".join("%.4f" % s for s in scores)))

## 11 · Що знає кожен шар

Поставимо лінійний зонд окремо на кожен шар. Вектор шару усереднюємо по непорожніх
позиціях, зонд — звичайна логістична регресія, доведена до збіжності, три зерна.

Заразом перевіримо дві речі. По-перше, що дає вектор `[CLS]` **до** першого шару
енкодера. По-друге, скільки з цього дає **ненавчена** модель — це контроль, без
якого будь-яке число зонда не має з чим порівнюватись.

In [ ]:
def layer_vectors(encoder, sentences):
    """Для кожного шару — усереднений по позиціях вектор; окремо [CLS] нульового шару."""
    encoder.eval()
    stacks = [[] for _ in range(LAYERS + 1)]
    cls_zero = []
    with torch.no_grad():
        for i in range(0, len(sentences), 256):
            x = pad_batch(sentences[i:i + 256])
            stages = encoder(x, collect=True)
            weight = (x != PAD).unsqueeze(-1).float()
            for k in range(LAYERS + 1):
                stacks[k].append(((stages[k] * weight).sum(1) / weight.sum(1)).numpy())
            cls_zero.append(stages[0][:, 0].numpy())
    return [np.concatenate(s) for s in stacks], np.concatenate(cls_zero)

def converged_probe(features_train, y_train, features_test, y_test, n_labels=2000,
                    seeds=3, offset=0):
    """Логістична регресія на готових ознаках: задача опукла, зерно міняє лише вибірку."""
    out = []
    for seed in range(seeds):
        rng = np.random.default_rng(offset + seed)
        chosen = rng.choice(len(features_train), n_labels, replace=False)
        model = LogisticRegression(max_iter=1000, class_weight="balanced")
        model.fit(features_train[chosen], y_train[chosen])
        out.append(f1_error(model.predict(features_test).tolist(), y_test))
    return out

probe_train_x = train_x[:6000]
probe_train_y = np.array(train_y[:6000])
layers_train, cls_train = layer_vectors(body, probe_train_x)
layers_test,  cls_test  = layer_vectors(body, TEST_EVAL[0])

for k in range(LAYERS + 1):
    scores = converged_probe(layers_train[k], probe_train_y, layers_test[k], TEST_EVAL[1])
    print("шар %d: медіана %.4f  купа %.4f…%.4f" % (k, np.median(scores), min(scores), max(scores)))

scores = converged_probe(cls_train, probe_train_y, cls_test, TEST_EVAL[1])
print("[CLS] до першого шару енкодера: %.4f  — той самий вектор для всіх речень"
      % np.median(scores))

In [ ]:
torch.manual_seed(0)
untrained = Encoder(VOCAB)            # та сама архітектура, але без передтренування
raw_train, _ = layer_vectors(untrained, probe_train_x)
raw_test, _ = layer_vectors(untrained, TEST_EVAL[0])
raw_scores = converged_probe(raw_train[2], probe_train_y, raw_test[2], TEST_EVAL[1])
top_scores = converged_probe(layers_train[2], probe_train_y, layers_test[2], TEST_EVAL[1])
print("зонд на передтренованому тілі: медіана %.4f  купа %.4f…%.4f"
      % (np.median(top_scores), min(top_scores), max(top_scores)))
print("зонд на НЕНАВЧЕНОМУ тілі     : медіана %.4f  купа %.4f…%.4f"
      % (np.median(raw_scores), min(raw_scores), max(raw_scores)))
overlap = not (min(top_scores) > max(raw_scores) or min(raw_scores) > max(top_scores))
print("купи %s" % ("перетинаються — виграшу передтренування не видно"
                   if overlap else "НЕ перетинаються — передтренування допомогло"))
del untrained, raw_train, raw_test
_ = gc.collect()
print("(контрольну модель прибрано з памʼяті)")

Другий погляд на шари — не через задачу, а через самі вектори. Візьмемо сорок
найчастіших слів і порахуємо, наскільки схожі між собою вектори **того самого слова**
в різних реченнях. Поруч — середній косинус між **різними** словами.

Дивитись на перше число окремо не можна: у векторів буває спільний напрямок, у який
дивляться всі, і він завищує будь-який косинус. Змістовна тут **різниця** — вона й
каже, скільки у векторі лишилось від того, яке це слово.

In [ ]:
def word_similarity(encoder, sentences, n_words=40, per_word=30):
    banks = [collections.defaultdict(list) for _ in range(LAYERS + 1)]
    encoder.eval()
    with torch.no_grad():
        for i in range(0, len(sentences), 256):
            x = pad_batch(sentences[i:i + 256])
            stages = encoder(x, collect=True)
            real = (x != PAD) & (x != CLS)
            token_ids = x[real].tolist()
            for k in range(LAYERS + 1):
                vectors = stages[k][real].numpy()
                for j, token in enumerate(token_ids):
                    banks[k][token].append(vectors[j])
    frequent = [t for t, _ in collections.Counter(
        {t: len(v) for t, v in banks[0].items()}).most_common(n_words)]
    rows = []
    for k in range(LAYERS + 1):
        same_word, one_each = [], []
        for token in frequent:
            block = np.array(banks[k][token][:per_word])
            block = block / np.linalg.norm(block, axis=1, keepdims=True)
            gram = block @ block.T
            m = len(block)
            same_word.append((gram.sum() - m) / (m * m - m))
            one_each.append(block[0])
        pool = np.array(one_each)
        gram = pool @ pool.T
        m = len(pool)
        different = (gram.sum() - m) / (m * m - m)
        rows.append((float(np.mean(same_word)), float(different)))
    return rows

print("%-6s %-16s %-16s %s" % ("шар", "те саме слово", "різні слова", "розрив"))
for k, (same, different) in enumerate(word_similarity(body, test_x[:3000])):
    print("%-6d %-16.4f %-16.4f %.4f" % (k, same, different, same - different))

## 12 · Збіжний зонд: якість від кількості міток

Заморожене тіло має властивість, якої немає в решти двох способів: вектори рахуються
один раз, а далі лишається звичайна лінійна класифікація — задача **опукла**, тобто з
одним мінімумом. Її не треба вчити ста кроками, її можна розвʼязати точно.

Тому подивімось, що дає зонд, доведений до збіжності, на різній кількості міток.

In [ ]:
for n_labels in (200, 500, 2000, 6000):
    scores = converged_probe(layers_train[2], probe_train_y, layers_test[2],
                             TEST_EVAL[1], n_labels=n_labels, offset=100)
    print("%5d міток: медіана %.4f  купа %.4f…%.4f  ширина %.4f"
          % (n_labels, np.median(scores), min(scores), max(scores),
             max(scores) - min(scores)))
del layers_train, layers_test, cls_train, cls_test
_ = gc.collect()
print("(вектори шарів прибрано з памʼяті)")

## 13 · Рубіж без нейромереж

Останнє й найважливіше: а чи варта вся ця конструкція заходу? Поставимо поруч
TF-IDF із логістичною регресією — те, що курс уміє з теми 06. Сталу регуляризації
добираємо на **відкладеній** вибірці, як і крок нейромережам: базу треба вичавити
щонайменше так само старанно, як нову модель.

In [ ]:
texts = [" ".join(words) for words, _ in rows]
train_text, hold_text, test_text = texts[:cut_train], texts[cut_train:cut_hold], texts[cut_hold:]

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\S+", min_df=2)
X_train = vectorizer.fit_transform(train_text)
X_hold = vectorizer.transform(hold_text[:1000])
X_test = vectorizer.transform(test_text[:3000])

best_c, best_score = None, -1
for c in (1, 4, 16, 64, 256):
    model = LogisticRegression(max_iter=1000, C=c, class_weight="balanced").fit(X_train, np.array(train_y))
    score = f1_error(model.predict(X_hold).tolist(), hold_y[:1000])
    print("  C=%-5g відкладена F1 %.4f" % (c, score))
    if score > best_score:
        best_score, best_c = score, c
print("  -> беремо C=%g\n" % best_c)

model = LogisticRegression(max_iter=1000, C=best_c, class_weight="balanced").fit(X_train, np.array(train_y))
tfidf_all = f1_error(model.predict(X_test).tolist(), TEST_EVAL[1])
print("TF-IDF + логістична, усі %d міток : F1 %.4f" % (len(train_text), tfidf_all))

tfidf_small = []
for seed in range(5):
    rng = np.random.default_rng(200 + seed)
    chosen = rng.choice(len(train_text), LABELS, replace=False)
    small_vec = TfidfVectorizer(token_pattern=r"(?u)\S+", min_df=2)
    Xs = small_vec.fit_transform([train_text[i] for i in chosen])
    model = LogisticRegression(max_iter=1000, C=best_c, class_weight="balanced").fit(
        Xs, np.array(train_y)[chosen])
    tfidf_small.append(f1_error(model.predict(small_vec.transform(test_text[:3000])).tolist(),
                                TEST_EVAL[1]))
print("TF-IDF + логістична, ті самі %d міток: медіана %.4f  купа %.4f…%.4f"
      % (LABELS, np.median(tfidf_small), min(tfidf_small), max(tfidf_small)))

## 14 · Підсумкова таблиця

Складімо все в одну таблицю — і зверни увагу на останній рядок.

In [ ]:
print("%-34s %10s %10s %-18s" % ("спосіб", "ваг", "медіана", "купа по зернах"))
print("-" * 78)
for mode, title in (("probe", "лінійний зонд"),
                    ("partial", "часткове донавчання"),
                    ("full", "повне донавчання")):
    m = main[mode]
    print("%-34s %10d %10.4f  %.4f…%.4f"
          % (title, m["weights"], m["median"], m["lo"], m["hi"]))
print("%-34s %10s %10.4f  %.4f…%.4f"
      % ("TF-IDF + логістична, 2000 міток", "—", np.median(tfidf_small),
         min(tfidf_small), max(tfidf_small)))
print("%-34s %10s %10.4f  %-18s"
      % ("TF-IDF + логістична, усі мітки", "—", tfidf_all, "один прогін"))
print()
best_net = max(main[m]["median"] for m in main)
print("найкращий зі способів переносу: %.4f" % best_net)
print("TF-IDF на тих самих 2000 мітках: %.4f" % np.median(tfidf_small))
print("різниця: %+.4f на користь %s"
      % (np.median(tfidf_small) - best_net,
         "лічильників" if np.median(tfidf_small) > best_net else "переносу"))
print()
print("процесорного часу на весь зошит: %.1f с" % (time.process_time() - START))

## 15 · Завдання

**🟢 Рівень 1.** Додай до сітки з розділу 7 ще два значення швидкості навчання —
`3e-5` і `3e-2` — і перевір, чи не поїхав вибір для якогось зі способів. Якщо поїхав,
скажи, у який бік і чому.

**🟡 Рівень 2.** Заміряй забування не лише для повного донавчання, а й для часткового:
поверни `measure_mlm=True` у виклику для `partial` і побудуй ту саму таблицю. Чи
береже часткове донавчання мову краще за повне — і на скільки?

**🔴 Рівень 3.** Зроби голову для **розмітки токенів**: замість `classifier(h[:, 0])`
постав `classifier(h)` і навчи модель ставити мітку кожному токену. Мітку візьми
просту й перевірювану: «це слово трапляється у словнику рідше за сто разів». Порівняй,
скільки ваг має така голова проти голови класифікації речення, і скажи, що змінилось
у тілі — нічого чи щось.